# 02. Segmentation

In [4]:
import sys
# sys.path.insert(0, '../src')
from src.data_loader import AAMI_MAP, CLASS_NAMES, CLASS_TO_IDX, WINDOW, DS1, DS2, segment_record
import wfdb
import numpy as np

In [2]:
beats, labels = segment_record('../data/raw/100')
print(beats.shape)
print(labels.shape)
print(set(labels))

(2271, 360)
(2271,)
{np.int64(0), np.int64(1), np.int64(2)}


Loop over all 48 records

In [3]:
from pathlib import Path

data_dir = Path('../data/raw')
all_beats, all_labels, all_record_ids = [], [], []

for hea_file in sorted(data_dir.glob('*.hea')):
    rec_id = hea_file.stem
    beats, labels = segment_record(str(data_dir / rec_id))
    all_beats.append(beats)
    all_labels.append(labels)
    all_record_ids.extend([rec_id] * len(beats))

all_beats  = np.concatenate(all_beats)
all_labels = np.concatenate(all_labels)

print(all_beats.shape)
print(all_labels.shape)
print(f'Total beats: {len(all_beats):,}')

(109438, 360)
(109438,)
Total beats: 109,438


## Train-Test Split

In [ ]:
def load_dataset(data_dir: str):
    ''' Load and segment records from DS1 and DS2, return train/test splits.'''
    train_beats, train_labels = [], []
    test_beats,  test_labels  = [], []

    for rec_id in DS1:
        beats, labels = segment_record(str(Path(data_dir) / rec_id))
        # append to train lists
        train_beats.append(beats)
        train_labels.append(labels)

    for rec_id in DS2:
        beats, labels = segment_record(str(Path(data_dir) / rec_id))
        # append to test lists
        test_beats.append(beats)
        test_labels.append(labels)

    return (
        np.concatenate(train_beats), np.concatenate(train_labels),
        np.concatenate(test_beats),  np.concatenate(test_labels)
    )

In [9]:
X_train, y_train, X_test, y_test = load_dataset('../data/raw')
print(X_train.shape, y_train.shape)
print(X_test.shape,  y_test.shape)

(50995, 360) (50995,)
(49687, 360) (49687,)


Let's check the class distribution

In [10]:
from collections import Counter

print('Train:', Counter(y_train.tolist()))
print('Test: ', Counter(y_test.tolist()))

Train: Counter({0: 45841, 2: 3788, 1: 944, 3: 414, 4: 8})
Test:  Counter({0: 44235, 2: 3220, 1: 1837, 3: 388, 4: 7})


In [11]:
# Save the prcessed data for later use
np.savez(
    '../data/processed/beats_split.npz',
    X_train=X_train, y_train=y_train,
    X_test=X_test,   y_test=y_test
)
print('Saved.')

Saved.
